# LeetCode 313 — Super Ugly Number bằng Python

Link đề: https://leetcode.com/problems/super-ugly-number/ · Độ khó: **Medium** · Chủ đề: Array, Math, Dynamic Programming, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def nthSuperUglyNumber(self, n: int, primes: List[int]) -> int:
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu **số siêu xấu (super ugly number)**: số nguyên dương mà **mọi thừa số nguyên tố đều nằm trong tập `primes` cho trước**; `1` luôn là số siêu xấu (tích rỗng). Tìm số siêu xấu thứ `n` theo thứ tự tăng dần.
2. Thấy cấu trúc "**mỗi số xấu = số xấu trước đó × một số nguyên tố**" và đi từ **duyệt từng số tự nhiên** → **min-heap có khử trùng bằng `set`** → **min-heap + mẹo `x % p == 0: break` khử trùng không cần `set`** → **DP nhiều con trỏ (mở rộng bài Ugly Number II)** → **heap `k` luồng (gộp `k` dãy đã sắp)**.
3. Hiểu **vì sao mẹo `break` khi `x % p == 0` sinh mỗi số đúng một lần** (giống ý tưởng sàng tuyến tính).
4. Biết cách **đo và so sánh độ phức tạp** `O(n·k)`, `O(n·k·log)` và `O(n·log k)`.
5. Tránh lỗi kinh điển: heap không khử trùng, chỉ tăng con trỏ đầu tiên khi bằng nhau, bỏ nhầm các số khi dùng `continue` thay `break`, quên hạt giống `ugly[0] = 1`.

## 2. Đề bài & ví dụ

**Số siêu xấu** là số nguyên dương mà **mọi thừa số nguyên tố** của nó đều nằm trong mảng `primes`.

Cho số nguyên `n` và mảng số nguyên `primes`, trả về **số siêu xấu thứ `n`**. Đề bảo đảm đáp án vừa trong số nguyên có dấu 32 bit.

**Ví dụ 1:** `n = 12`, `primes = [2,7,13,19]` → `32`. Dãy `12` số siêu xấu đầu: `[1,2,4,7,8,13,14,16,19,26,28,32]`.

**Ví dụ 2:** `n = 1`, `primes = [2,3,5]` → `1` (`1` không có thừa số nguyên tố nào nên luôn là số siêu xấu).

**Ràng buộc:** `1 <= n <= 10⁵`; `1 <= primes.length <= 100`; `2 <= primes[i] <= 1000`; `primes[i]` đảm bảo là số nguyên tố; mọi giá trị trong `primes` **khác nhau** và được sắp tăng dần.

In [1]:
from typing import List
import random
import heapq

examples = [
    (12, [2, 7, 13, 19], 32),
    (1, [2, 3, 5], 1),
]
for n, primes, e in examples:
    print("n =", n, "| primes =", primes, "-> kỳ vọng", e)

n = 12 | primes = [2, 7, 13, 19] -> kỳ vọng 32
n = 1 | primes = [2, 3, 5] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Số siêu xấu sinh từ số siêu xấu nhỏ hơn

Nếu `x` là số siêu xấu và `p` thuộc `primes` thì `x · p` cũng là số siêu xấu. Ngược lại, mọi số siêu xấu `y > 1` đều có dạng `y = x · p` với `x` là số siêu xấu **nhỏ hơn** và `p ∈ primes` (lấy `p` là một thừa số nguyên tố của `y`, `x = y / p`). Nên toàn bộ dãy sinh ra từ `1` bằng phép nhân với `primes`.

In [2]:
primes = [2, 3, 5]
seen = {1}
frontier = [1]
for _ in range(3):
    frontier = sorted({x * p for x in frontier for p in primes} - seen)
    seen |= set(frontier)
print("các số siêu xấu <= 30 sinh ra sau 3 vòng:", sorted(x for x in seen if x <= 30))

các số siêu xấu <= 30 sinh ra sau 3 vòng: [1, 2, 3, 4, 5, 6, 8, 9, 10, 12, 15, 18, 20, 25, 27, 30]


### 3.2. Kiểm tra một số có phải siêu xấu không

Chia `x` liên tiếp cho các số nguyên tố trong `primes` đến khi không chia được; nếu còn lại `1` thì `x` là số siêu xấu. Cách này dùng làm **oracle** (đơn giản nhưng chậm khi đáp án lớn).

In [3]:
def is_super_ugly(x, primes):
    for p in primes:
        while x % p == 0:
            x //= p
    return x == 1


print([x for x in range(1, 40) if is_super_ugly(x, [2, 3, 5])])

[1, 2, 3, 4, 5, 6, 8, 9, 10, 12, 15, 16, 18, 20, 24, 25, 27, 30, 32, 36]


### 3.3. Min-heap (`heapq`) và gộp `k` dãy đã sắp

`heapq` là min-heap: `heappush`, `heappop` mất `O(log size)`. Gộp `k` dãy đã sắp là một bài heap kinh điển: giữ phần tử nhỏ nhất của mỗi dãy trong heap, lấy nhỏ nhất ra và thay bằng phần tử kế tiếp của chính dãy đó. Dãy thứ `j` ở đây là `{p_j · ugly[i] : i = 0, 1, ...}`.

In [4]:
a = [1, 4, 7]
b = [2, 5, 8]
c = [3, 6, 9]
print(list(heapq.merge(a, b, c)))      # [1, 2, 3, 4, 5, 6, 7, 8, 9]

[1, 2, 3, 4, 5, 6, 7, 8, 9]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

`primes = [2, 3, 5]`: dãy `1, 2, 3, 4, 5, 6, 8, 9, 10, 12, ...`.

- Số tiếp theo luôn là **số nhỏ nhất chưa dùng** trong tập `{ugly[i] · p}`.
- Ứng viên ban đầu: `1·2 = 2`, `1·3 = 3`, `1·5 = 5` → chọn `2`.
- Sau khi chọn `2`: ứng viên `2·2 = 4`, `1·3 = 3`, `1·5 = 5` → chọn `3`.
- Sau khi chọn `3`: ứng viên `2·2 = 4`, `2·3 = 6`, `1·5 = 5` → chọn `4`.

Mỗi số nguyên tố `p` có một **con trỏ** `idx[p]` chỉ vào số xấu mà nó sẽ nhân tiếp theo. Ứng viên của `p` là `ugly[idx[p]] · p`. Số nhỏ nhất trong các ứng viên là số tiếp theo; **mọi** `p` có ứng viên bằng số đó đều phải tăng con trỏ (nếu không sẽ trùng, ví dụ `6 = 2·3 = 3·2`).

In [5]:
primes = [2, 3, 5]
ugly = [1]
idx = [0, 0, 0]
for step in range(1, 8):
    cands = [ugly[idx[j]] * primes[j] for j in range(3)]
    nxt = min(cands)
    ugly.append(nxt)
    for j in range(3):
        if cands[j] == nxt:
            idx[j] += 1
    print(f"bước {step}: ứng viên {cands} -> chọn {nxt}, con trỏ {idx}")
print(ugly)

bước 1: ứng viên [2, 3, 5] -> chọn 2, con trỏ [1, 0, 0]
bước 2: ứng viên [4, 3, 5] -> chọn 3, con trỏ [1, 1, 0]
bước 3: ứng viên [4, 6, 5] -> chọn 4, con trỏ [2, 1, 0]
bước 4: ứng viên [6, 6, 5] -> chọn 5, con trỏ [2, 1, 1]
bước 5: ứng viên [6, 6, 10] -> chọn 6, con trỏ [3, 2, 1]
bước 6: ứng viên [8, 9, 10] -> chọn 8, con trỏ [4, 2, 1]
bước 7: ứng viên [10, 9, 10] -> chọn 9, con trỏ [4, 3, 1]
[1, 2, 3, 4, 5, 6, 8, 9]


## 5. Cách 1 — Brute Force / baseline: duyệt từng số tự nhiên

**Ý tưởng.** Duyệt `x = 1, 2, 3, ...`; số nào là số siêu xấu thì đếm, đến khi đếm tới `n` thì trả `x`. Chỉ khả thi khi đáp án nhỏ; dùng làm **oracle** cho các test nhỏ.

In [6]:
def nth_super_ugly_scan(n: int, primes: List[int]) -> int:
    count, x = 0, 0
    while count < n:
        x += 1
        if is_super_ugly(x, primes):
            count += 1
    return x


assert nth_super_ugly_scan(12, [2, 7, 13, 19]) == 32
assert nth_super_ugly_scan(1, [2, 3, 5]) == 1
assert nth_super_ugly_scan(10, [2, 3, 5]) == 12
print("PASS ví dụ (duyệt từng số)")

PASS ví dụ (duyệt từng số)


**Độ phức tạp (Cách 1).** Time `O(A · k)` với `A` là giá trị đáp án; `A` có thể rất lớn (với `primes = [2]` thì `A = 2ⁿ⁻¹`), nên **không khả thi** cho `n = 10⁵`. Space `O(1)`.

**Nút thắt.** Hầu hết số tự nhiên **không** là số siêu xấu, ta lãng phí kiểm tra. Cần **sinh trực tiếp** các số siêu xấu theo thứ tự.

## 6. Ý tưởng thuật toán / bước đột phá

Vì mỗi số siêu xấu (khác `1`) bằng `x · p` với `x` nhỏ hơn đã có, ta **sinh theo thứ tự tăng**:

1. **Heap:** giữ tập ứng viên trong min-heap; lấy nhỏ nhất `x`; đẩy `x · p` cho mọi `p`. Vấn đề: cùng một số có thể sinh nhiều lần (`6 = 2·3 = 3·2`) → cần **khử trùng**.
2. **Mẹo khử trùng:** với `x` lấy ra, đẩy `x · p` theo thứ tự `p` **tăng dần** và **dừng ngay sau khi gặp `p` chia hết `x`**. Khi đó mỗi số được sinh **duy nhất một cách** (từ cách phân tích có thừa số nguyên tố nhỏ nhất ở "đuôi").
3. **Con trỏ:** mỗi `p` giữ một con trỏ trên dãy đã sinh; số tiếp theo là min các ứng viên; tăng mọi con trỏ cùng ra giá trị min.

Đây là phép **gộp `k` dãy đã sắp** `{p · ugly[i]}`.

In [7]:
# Kiểm chứng mẹo break: mỗi số được sinh đúng một lần
def generate_with_break(limit, primes):
    heap = [1]
    produced = []
    while heap:
        x = heapq.heappop(heap)
        if x > limit:
            continue
        produced.append(x)
        for p in primes:
            heapq.heappush(heap, x * p)
            if x % p == 0:
                break
    return produced

produced = generate_with_break(300, [2, 3, 5, 7])
assert len(produced) == len(set(produced))                         # không trùng
assert sorted(produced) == [x for x in range(1, 301) if is_super_ugly(x, [2, 3, 5, 7])]
print("PASS: heap + break sinh mỗi số siêu xấu <= 300 đúng một lần:", len(produced), "số")

PASS: heap + break sinh mỗi số siêu xấu <= 300 đúng một lần: 82 số


## 7. Các cách tiếp theo

### Cách 2 — Min-heap + `set` để khử trùng

**Giải thích.** Heap chứa các ứng viên. Mỗi lần lấy `x`, đẩy `x · p` cho mọi `p` **chỉ khi chưa từng đẩy** (kiểm tra bằng `set seen`). Lấy ra `n` lần thì trả giá trị thứ `n`.

In [8]:
def nth_super_ugly_heap_set(n: int, primes: List[int]) -> int:
    heap = [1]
    seen = {1}
    x = 1
    for _ in range(n):
        x = heapq.heappop(heap)
        for p in primes:
            y = x * p
            if y not in seen:
                seen.add(y)
                heapq.heappush(heap, y)
    return x


for n, primes, e in examples:
    assert nth_super_ugly_heap_set(n, primes) == e
print("PASS ví dụ (heap + set)")

PASS ví dụ (heap + set)


**Độ phức tạp Cách 2:** Time `O(n·k·log(n·k))` (mỗi lần lấy ra đẩy tối đa `k` số); Space `O(n·k)` cho `set` và heap.

### Cách 3 — Min-heap + mẹo `x % p == 0: break` (cách của lời giải tham khảo)

**Giải thích.** Như Cách 2 nhưng không cần `set`: với mỗi `x` lấy ra, duyệt `p` tăng dần, đẩy `x · p`, và **thoát ngay** nếu `x % p == 0`. Vì `primes` đã sắp tăng dần và khác nhau nên mỗi số chỉ được đẩy đúng một lần.

In [9]:
def nth_super_ugly_heap_break(n: int, primes: List[int]) -> int:
    heap = [1]
    x = 1
    for _ in range(n):
        x = heapq.heappop(heap)
        for p in primes:
            heapq.heappush(heap, x * p)
            if x % p == 0:
                break
    return x


for n, primes, e in examples:
    assert nth_super_ugly_heap_break(n, primes) == e
print("PASS ví dụ (heap + break)")

PASS ví dụ (heap + break)


**Độ phức tạp Cách 3:** Time `O(n·k·log(n·k))` trong trường hợp xấu (mỗi `x` đẩy tối đa `k` số, trung bình ít hơn vì `break`); Space `O(n·k)` (heap) trong trường hợp xấu, thực tế ít hơn.

### Cách 4 — DP nhiều con trỏ (mở rộng Ugly Number II)

**Giải thích.** `ugly[0] = 1`. Mỗi `primes[j]` có con trỏ `idx[j]`. Mỗi bước: tính `nxt = min(ugly[idx[j]] · primes[j])`, thêm vào `ugly`, rồi **tăng mọi `idx[j]` mà ứng viên bằng `nxt`**. Không cần heap.

In [10]:
def nth_super_ugly_pointers(n: int, primes: List[int]) -> int:
    k = len(primes)
    ugly = [1] * n
    idx = [0] * k
    for i in range(1, n):
        nxt = min(ugly[idx[j]] * primes[j] for j in range(k))
        ugly[i] = nxt
        for j in range(k):
            if ugly[idx[j]] * primes[j] == nxt:
                idx[j] += 1
    return ugly[n - 1]


for n, primes, e in examples:
    assert nth_super_ugly_pointers(n, primes) == e
print("PASS ví dụ (nhiều con trỏ)")

PASS ví dụ (nhiều con trỏ)


**Độ phức tạp Cách 4:** Time `O(n·k)`; Space `O(n + k)`. Với `n = 10⁵`, `k = 100` là `10⁷` phép toán: đủ cho giới hạn của đề.

### Cách 5 — Heap `k` luồng (gộp `k` dãy đã sắp), khử trùng khi lấy ra

**Giải thích.** Heap chỉ có đúng `k` phần tử `(giá trị, p, con_trỏ)`: ứng viên hiện tại của mỗi luồng. Lấy phần tử nhỏ nhất; nếu khác số xấu vừa thêm thì thêm vào dãy (khử trùng); rồi đẩy ứng viên kế của **chính luồng đó**: `p · ugly[con_trỏ + 1]`. Mỗi lần lấy ra chỉ đẩy **một** phần tử.

In [11]:
def nth_super_ugly_kway(n: int, primes: List[int]) -> int:
    ugly = [1]
    heap = [(p, p, 0) for p in primes]
    heapq.heapify(heap)
    while len(ugly) < n:
        val, p, i = heapq.heappop(heap)
        if val != ugly[-1]:                       # khử trùng
            ugly.append(val)
        heapq.heappush(heap, (p * ugly[i + 1], p, i + 1))
    return ugly[n - 1]


for n, primes, e in examples:
    assert nth_super_ugly_kway(n, primes) == e
print("PASS ví dụ (heap k luồng)")

PASS ví dụ (heap k luồng)


**Độ phức tạp Cách 5:** Time `O((n + k)·log k)` (có thể phải lấy ra nhiều hơn `n` lần do trùng, nhưng mỗi số xấu bị lấy tối đa `k` lần: `O(n·k·log k)` xấu nhất, thực tế gần `O(n·log k)` vì trùng ít); Space `O(n + k)`.

> Cách 4 là cách nộp bài (gọn, đủ nhanh, chắc chắn đúng). Cách 3 là cách tham khảo ngắn. Cách 5 mạnh nhất khi `k` lớn. Cách 1 dùng làm oracle.

## 8. TRACE từng bước

Theo dõi Cách 4 trên `n = 12`, `primes = [2, 7, 13, 19]`: in ứng viên của mỗi số nguyên tố, giá trị chọn và con trỏ.

In [12]:
def trace_pointers(n, primes):
    k = len(primes)
    ugly = [1]
    idx = [0] * k
    print(f"ugly[0] = 1, idx = {idx}")
    for i in range(1, n):
        cands = [ugly[idx[j]] * primes[j] for j in range(k)]
        nxt = min(cands)
        ugly.append(nxt)
        for j in range(k):
            if cands[j] == nxt:
                idx[j] += 1
        print(f"ugly[{i:2}] = {nxt:3}  | ứng viên {cands} -> idx = {idx}")
    return ugly[n - 1]


assert trace_pointers(12, [2, 7, 13, 19]) == 32

ugly[0] = 1, idx = [0, 0, 0, 0]
ugly[ 1] =   2  | ứng viên [2, 7, 13, 19] -> idx = [1, 0, 0, 0]
ugly[ 2] =   4  | ứng viên [4, 7, 13, 19] -> idx = [2, 0, 0, 0]
ugly[ 3] =   7  | ứng viên [8, 7, 13, 19] -> idx = [2, 1, 0, 0]
ugly[ 4] =   8  | ứng viên [8, 14, 13, 19] -> idx = [3, 1, 0, 0]
ugly[ 5] =  13  | ứng viên [14, 14, 13, 19] -> idx = [3, 1, 1, 0]
ugly[ 6] =  14  | ứng viên [14, 14, 26, 19] -> idx = [4, 2, 1, 0]
ugly[ 7] =  16  | ứng viên [16, 28, 26, 19] -> idx = [5, 2, 1, 0]
ugly[ 8] =  19  | ứng viên [26, 28, 26, 19] -> idx = [5, 2, 1, 1]
ugly[ 9] =  26  | ứng viên [26, 28, 26, 38] -> idx = [6, 2, 2, 1]
ugly[10] =  28  | ứng viên [28, 28, 52, 38] -> idx = [7, 3, 2, 1]
ugly[11] =  32  | ứng viên [32, 49, 52, 38] -> idx = [8, 3, 2, 1]


Dãy tạo ra: `1, 2, 4, 7, 8, 13, 14, 16, 19, 26, 28, 32`.

Ví dụ một bước: khi `ugly = [1, 2, 4, 7, 8, 13]` và `idx = [3, 1, 1, 0]`, ứng viên là `[ugly[3]·2, ugly[1]·7, ugly[1]·13, ugly[0]·19] = [14, 14, 26, 19]`; giá trị nhỏ nhất `14` xuất hiện ở hai số nguyên tố (`7·2 = 2·7`) nên **cả hai** con trỏ cùng tăng (khử trùng).

## 9. Code nộp LeetCode

Bản nộp dùng Cách 4 (DP nhiều con trỏ), đúng chữ ký của LeetCode.

In [13]:
class Solution:
    def nthSuperUglyNumber(self, n: int, primes: List[int]) -> int:
        k = len(primes)
        ugly = [1] * n
        idx = [0] * k
        for i in range(1, n):
            nxt = min(ugly[idx[j]] * primes[j] for j in range(k))
            ugly[i] = nxt
            for j in range(k):
                if ugly[idx[j]] * primes[j] == nxt:
                    idx[j] += 1
        return ugly[n - 1]


for n, primes, e in examples:
    assert Solution().nthSuperUglyNumber(n, primes) == e
print("PASS ví dụ đề cho")

PASS ví dụ đề cho


## 10. Vì sao thuật toán đúng

- **Đầy đủ và đúng thứ tự (con trỏ):** mỗi số nguyên tố `p` có dãy ứng viên `p · ugly[0] < p · ugly[1] < ...` tăng dần. Số siêu xấu thứ `i` là phần tử nhỏ nhất **chưa dùng** trong hợp của `k` dãy này. Con trỏ `idx[j]` trỏ vào phần tử đầu tiên của dãy `j` chưa nhỏ hơn hoặc bằng `ugly[i − 1]`.
- **Khử trùng:** nếu nhiều dãy cùng có ứng viên bằng giá trị vừa chọn, **tất cả** phải tăng con trỏ; nếu chỉ tăng một thì ở bước sau các dãy còn lại vẫn đưa ra cùng giá trị cũ → trùng.
- **Không bỏ sót:** mọi số siêu xấu `y > 1` bằng `p · x` với `x < y` siêu xấu, nên `y` nằm trong dãy `p`, và con trỏ chỉ vượt qua `y` khi `y` đã được chọn.
- **Mẹo `break` (Cách 3):** với `x` lấy ra, ta đẩy `x · p` cho các `p` từ nhỏ đến lớn và dừng sau khi gặp `p` đầu tiên chia hết `x`; tức là chỉ đẩy `x · p` với `p ≤ spf(x)` (`spf(x)` = số nguyên tố nhỏ nhất trong `primes` chia hết `x`; với `x = 1` thì đẩy với mọi `p`). Mỗi số siêu xấu `y > 1` được sinh từ **đúng một** `x`: chính là `x = y / spf(y)` (lấy `p = spf(y)`; mọi cách viết `y = x · p` khác có `p > spf(y)` bị loại). Do đó không trùng, không sót.
- **Cách 5:** mỗi luồng sinh dãy tăng; gộp `k` luồng đã sắp bằng heap cho dãy tăng; khử trùng khi giá trị lấy ra bằng phần tử cuối vừa thêm.

## 11. Phân tích Time / Space Complexity

- **Cách 1:** Time `O(A · k)` với `A` là đáp án (có thể cực lớn); Space `O(1)`.
- **Cách 2:** Time `O(n·k·log(n·k))`; Space `O(n·k)`.
- **Cách 3:** Time `O(n·k·log(n·k))` xấu nhất; Space `O(n·k)` xấu nhất.
- **Cách 4:** Time `O(n·k)`; Space `O(n + k)`.
- **Cách 5:** Time `O(n·k·log k)` xấu nhất, thường gần `O(n·log k)`; Space `O(n + k)`.
- Với `n = 10⁵`, `k = 100`: Cách 4 ≈ `10⁷` phép toán; Cách 5 ít hơn đáng kể.

## 12. Bộ test

Đối chiếu mọi cách nhanh với oracle duyệt từng số (khi đáp án nhỏ), và với nhau (khi đáp án lớn). `primes` luôn **khác nhau và tăng dần**, như đề yêu cầu.

In [14]:
impls = {
    "heap_set": nth_super_ugly_heap_set,
    "heap_break": nth_super_ugly_heap_break,
    "pointers": nth_super_ugly_pointers,
    "kway": nth_super_ugly_kway,
    "submit": lambda n, primes: Solution().nthSuperUglyNumber(n, primes),
}
SMALL_PRIMES = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]
ALL_PRIMES = [p for p in range(2, 1001) if all(p % d for d in range(2, int(p ** 0.5) + 1))]
assert len(ALL_PRIMES) == 168


def check(n, primes, expected=None, names=None, scan=True):
    base = nth_super_ugly_pointers(n, primes)
    if expected is not None:
        assert base == expected, (n, primes, base, expected)
    if scan and base <= 20000:
        assert nth_super_ugly_scan(n, primes) == base, (n, primes)
    for name, fn in impls.items():
        if names is not None and name not in names:
            continue
        assert fn(n, list(primes)) == base, (name, n, primes)
    return base


for n, primes, e in examples:
    check(n, primes, e)
print("PASS 2 ví dụ của đề")

PASS 2 ví dụ của đề


### 12.1. Các trường hợp biên và giá trị tính tay

In [15]:
cases = [
    (1, [2], 1, "n = 1"),
    (5, [2], 16, "một số nguyên tố: lũy thừa của 2"),
    (7, [3], 729, "một số nguyên tố: 3^6"),
    (10, [2, 3, 5], 12, "ugly number cổ điển, số thứ 10"),
    (5, [3, 5], 15, "dãy 1, 3, 5, 9, 15"),
    (6, [3, 5], 25, "dãy 1, 3, 5, 9, 15, 25"),
    (12, [2, 7, 13, 19], 32, "ví dụ của đề"),
    (2, [997], 997, "số nguyên tố lớn nhất dưới 1000"),
    (3, [2, 3], 3, "dãy 1, 2, 3"),
    (8, [2, 3], 12, "dãy 1, 2, 3, 4, 6, 8, 9, 12"),
]
for n, primes, e, why in cases:
    assert check(n, primes, e, scan=(e < 5000)) == e
    print("PASS", why)

PASS n = 1
PASS một số nguyên tố: lũy thừa của 2
PASS một số nguyên tố: 3^6
PASS ugly number cổ điển, số thứ 10
PASS dãy 1, 3, 5, 9, 15
PASS dãy 1, 3, 5, 9, 15, 25
PASS ví dụ của đề
PASS số nguyên tố lớn nhất dưới 1000
PASS dãy 1, 2, 3
PASS dãy 1, 2, 3, 4, 6, 8, 9, 12


### 12.2. Tính chất và phản ví dụ

In [16]:
rnd = random.Random(313)


def super_ugly_list(n, primes):
    ugly = [1]
    idx = [0] * len(primes)
    while len(ugly) < n:
        cands = [ugly[idx[j]] * primes[j] for j in range(len(primes))]
        nxt = min(cands)
        ugly.append(nxt)
        for j in range(len(primes)):
            if cands[j] == nxt:
                idx[j] += 1
    return ugly


# Tính chất: dãy tăng ngặt, bắt đầu bằng 1, mọi phần tử là số siêu xấu, và đúng là TẤT CẢ số siêu xấu tới phần tử cuối
for _ in range(100):
    primes = sorted(rnd.sample(SMALL_PRIMES, rnd.randint(1, 4)))
    n = rnd.randint(1, 60)
    seq = super_ugly_list(n, primes)
    assert seq[0] == 1 and all(a < b for a, b in zip(seq, seq[1:]))
    assert all(is_super_ugly(x, primes) for x in seq)
    if seq[-1] <= 5000:
        assert seq == [x for x in range(1, seq[-1] + 1) if is_super_ugly(x, primes)]
print("PASS tính chất: tăng ngặt, mọi phần tử đúng là số siêu xấu, không bỏ sót")

# Tính chất: thêm một số nguyên tố vào primes không làm đáp án lớn hơn
for _ in range(100):
    primes = sorted(rnd.sample(SMALL_PRIMES, rnd.randint(1, 4)))
    extra = sorted(set(primes) | {rnd.choice(SMALL_PRIMES)})
    n = rnd.randint(1, 80)
    assert nth_super_ugly_pointers(n, extra) <= nth_super_ugly_pointers(n, primes)
print("PASS tính chất: thêm số nguyên tố -> đáp án không tăng")

# Phản ví dụ: 6 = 2*3 = 3*2 xuất hiện hai cách => nếu không khử trùng thì thừa số
print("6 sinh ra từ 3*2 và 2*3:", 3 * 2 == 2 * 3 == 6)

PASS tính chất: tăng ngặt, mọi phần tử đúng là số siêu xấu, không bỏ sót
PASS tính chất: thêm số nguyên tố -> đáp án không tăng
6 sinh ra từ 3*2 và 2*3: True


### 12.3. Đối chiếu ngẫu nhiên (có seed) với brute force

In [17]:
rnd = random.Random(2026)
for trial in range(500):                                  # nhỏ: có oracle duyệt từng số
    k = rnd.randint(1, 5)
    primes = sorted(rnd.sample(SMALL_PRIMES, k))
    n = rnd.randint(1, 40)
    check(n, primes)

for trial in range(60):                                   # vừa: so các cách nhanh với nhau
    k = rnd.randint(1, 30)
    primes = sorted(rnd.sample(ALL_PRIMES, k))
    n = rnd.randint(1, 600)
    check(n, primes, scan=False)

# lớn theo đề: n = 10^5, k = 100 (chỉ cách nhanh)
primes = ALL_PRIMES[:100]
n = 10 ** 5
r1 = nth_super_ugly_heap_break(n, primes)
r2 = nth_super_ugly_kway(n, primes)
r3 = nth_super_ugly_pointers(n, primes)
assert r1 == r2 == r3
print("PASS 500 test nhỏ (oracle duyệt), 60 test vừa, 1 test n = 10^5 với 100 số nguyên tố:", r1)

PASS 500 test nhỏ (oracle duyệt), 60 test vừa, 1 test n = 10^5 với 100 số nguyên tố: 284445


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Heap không khử trùng

In [18]:
def sai_heap_khong_khu_trung(n, primes):
    heap = [1]
    x = 1
    for _ in range(n):
        x = heapq.heappop(heap)
        for p in primes:
            heapq.heappush(heap, x * p)                # sai: 6 = 2*3 = 3*2 được đẩy hai lần
    return x


print("không khử trùng, n = 7 ->", sai_heap_khong_khu_trung(7, [2, 3, 5]), "| đúng: 8")
assert sai_heap_khong_khu_trung(7, [2, 3, 5]) != 8

không khử trùng, n = 7 -> 6 | đúng: 8


**Vì sao sai:** `6` được sinh từ `3·2` và từ `2·3`, nên xuất hiện hai lần trong heap và được đếm hai lần; ra số `6` thay vì `8` ở vị trí thứ 7. Phải khử trùng bằng `set` hoặc mẹo `break`.

### Lỗi 2: Chỉ tăng con trỏ đầu tiên khi các ứng viên bằng nhau

In [19]:
def sai_mot_con_tro(n, primes):
    k = len(primes)
    ugly = [1] * n
    idx = [0] * k
    for i in range(1, n):
        nxt = min(ugly[idx[j]] * primes[j] for j in range(k))
        ugly[i] = nxt
        for j in range(k):
            if ugly[idx[j]] * primes[j] == nxt:
                idx[j] += 1
                break                                    # sai: các con trỏ khác cùng giá trị cũng phải tăng
    return ugly[n - 1]


print("chỉ tăng một con trỏ, n = 7 ->", sai_mot_con_tro(7, [2, 3, 5]), "| đúng: 8")
assert sai_mot_con_tro(7, [2, 3, 5]) != 8

chỉ tăng một con trỏ, n = 7 -> 6 | đúng: 8


**Vì sao sai:** ở bước chọn `6`, cả con trỏ của `2` và của `3` đều cho ứng viên `6`. Nếu chỉ tăng một thì bước sau con trỏ còn lại vẫn đưa ra `6`: dãy bị trùng.

### Lỗi 3: Dùng `continue` thay cho `break` trong mẹo khử trùng

In [20]:
def sai_continue(n, primes):
    heap = [1]
    x = 1
    for _ in range(n):
        x = heapq.heappop(heap)
        for p in primes:
            if x % p == 0:
                continue                                  # sai: phải đẩy x*p rồi mới break
            heapq.heappush(heap, x * p)
    return x


print("continue thay break, n = 4 ->", sai_continue(4, [2, 3, 5]), "| đúng: 4")
assert sai_continue(4, [2, 3, 5]) != 4

continue thay break, n = 4 -> 5 | đúng: 4


**Vì sao sai:** với `x = 2`, bản sai bỏ qua `p = 2` và mất số `4 = 2·2`; mọi lũy thừa của số nguyên tố đều bị mất. Đúng là: **đẩy `x · p` trước, rồi** `if x % p == 0: break`.

### Lỗi 4: Quên khởi tạo `ugly[0] = 1`

In [21]:
def sai_quen_so_1(n, primes):
    k = len(primes)
    ugly = [0] * n                                      # sai: phải ugly[0] = 1
    idx = [0] * k
    for i in range(1, n):
        nxt = min(ugly[idx[j]] * primes[j] for j in range(k))
        ugly[i] = nxt
        for j in range(k):
            if ugly[idx[j]] * primes[j] == nxt:
                idx[j] += 1
    return ugly[n - 1]


print("quên ugly[0] = 1 ->", sai_quen_so_1(10, [2, 3, 5]), "| đúng: 12")
assert sai_quen_so_1(10, [2, 3, 5]) != 12

quên ugly[0] = 1 -> 0 | đúng: 12


**Vì sao sai:** mọi số siêu xấu sinh từ `1` bằng phép nhân; nếu hạt giống là `0` thì mọi tích đều `0` và cả dãy toàn `0`. `1` (tích rỗng) là phần tử đầu tiên bắt buộc.

## 14. Mẹo làm nhanh & nhận dạng pattern

- "Số thứ `n` có thừa số nguyên tố thuộc tập cho trước" → **sinh theo thứ tự**: heap hoặc **nhiều con trỏ**; không duyệt từng số.
- Ugly Number II (264) là trường hợp `primes = [2, 3, 5]`; nên nhớ lời giải nhiều con trỏ và mở rộng thành `k` con trỏ.
- Khử trùng: tăng **mọi** con trỏ có ứng viên bằng giá trị vừa chọn; hoặc `set`; hoặc `break` khi `x % p == 0` với `primes` đã sắp.
- `k` lớn, `n` lớn → heap `k` luồng (`O(n·log k)`) thay vì quét `k` con trỏ mỗi bước.
- Đáp án có thể rất lớn nhưng Python không tràn số; ở ngôn ngữ khác phải chú ý tràn 32-bit.
- Test: oracle `is_super_ugly` chỉ dùng được khi đáp án nhỏ; còn lại so các cách với nhau.

## 15. Biến thể của bài

1. **Ugly Number I (263):** kiểm tra một số có phải ugly không.
2. **Ugly Number II (264):** `primes = [2, 3, 5]`.
3. **Liệt kê `n` số siêu xấu đầu tiên.**
4. **Đếm số siêu xấu `≤ N`.**
5. **Số siêu xấu thứ `n` trong một dải `[L, R]` / đáp án modulo.**

In [22]:
def is_ugly(x):
    """Biến thể 1 (263): thừa số chỉ 2, 3, 5."""
    return x > 0 and is_super_ugly(x, [2, 3, 5])


def nth_ugly(n):
    """Biến thể 2 (264)."""
    return nth_super_ugly_pointers(n, [2, 3, 5])


def first_n_super_ugly(n, primes):
    """Biến thể 3."""
    return super_ugly_list(n, primes)


def count_super_ugly_leq(N, primes):
    """Biến thể 4: đếm số siêu xấu <= N bằng BFS trên tập đã thấy."""
    if N < 1:
        return 0
    seen = {1}
    stack = [1]
    while stack:
        x = stack.pop()
        for p in primes:
            y = x * p
            if y <= N and y not in seen:
                seen.add(y)
                stack.append(y)
    return len(seen)


def count_super_ugly_in_range(L, R, primes):
    """Biến thể 5: đếm số siêu xấu trong [L, R]."""
    return count_super_ugly_leq(R, primes) - count_super_ugly_leq(L - 1, primes)


assert [x for x in range(1, 20) if is_ugly(x)] == [1, 2, 3, 4, 5, 6, 8, 9, 10, 12, 15, 16, 18]
assert nth_ugly(10) == 12 and nth_ugly(1) == 1
assert first_n_super_ugly(12, [2, 7, 13, 19]) == [1, 2, 4, 7, 8, 13, 14, 16, 19, 26, 28, 32]
random.seed(15)
for _ in range(100):
    primes = sorted(random.sample(SMALL_PRIMES, random.randint(1, 4)))
    N = random.randint(1, 500)
    assert count_super_ugly_leq(N, primes) == sum(is_super_ugly(x, primes) for x in range(1, N + 1))
    L = random.randint(1, N)
    assert count_super_ugly_in_range(L, N, primes) == sum(is_super_ugly(x, primes) for x in range(L, N + 1))
print("PASS biến thể: ugly I, II, liệt kê, đếm <= N, đếm trong đoạn")

PASS biến thể: ugly I, II, liệt kê, đếm <= N, đếm trong đoạn


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt từng số | kiểm tra phân tích thừa số | `O(A·k)` | `O(1)` | oracle, đáp án nhỏ |
| 2. Heap + `set` | khử trùng bằng tập đã thấy | `O(n·k·log(n·k))` | `O(n·k)` | dễ nghĩ |
| 3. Heap + `break` | khử trùng nhờ `x % p == 0` | `O(n·k·log(n·k))` | `O(n·k)` | gọn, cách tham khảo |
| 4. Nhiều con trỏ | min của `k` ứng viên | `O(n·k)` | `O(n + k)` | **cách nộp bài** |
| 5. Heap `k` luồng | gộp `k` dãy đã sắp | `O(n·k·log k)` xấu nhất | `O(n + k)` | `k` lớn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán `nth_super_ugly_pointers(10, [2, 3, 5])` và `nth_super_ugly_pointers(5, [2])`.

**Bài 2 (tính tay).** Liệt kê 6 số siêu xấu đầu tiên với `primes = [3, 5]`.

**Bài 3 (điền TODO).** Hoàn thành `nth_super_ugly_todo` theo Cách 4.

In [23]:
def nth_super_ugly_todo(n, primes):
    # TODO 1: ugly = [1] * n; idx = [0] * len(primes)
    # TODO 2: for i in range(1, n): nxt = min(ugly[idx[j]] * primes[j] for j)
    # TODO 3: ugly[i] = nxt
    # TODO 4: tăng MỌI idx[j] mà ugly[idx[j]] * primes[j] == nxt
    # TODO 5: trả ugly[n - 1]
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `nth_super_ugly_kway2(n, primes)` bằng `heapq.merge` hoặc heap `k` luồng, khử trùng khi giá trị lấy ra bằng phần tử cuối vừa thêm.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_leq(N, primes)` đếm số siêu xấu `≤ N`. (b) Viết `nth_super_ugly_bsearch(n, primes)` tìm số thứ `n` bằng **tìm kiếm nhị phân trên giá trị** kết hợp với `count_leq` (dùng cho `n`, `N` nhỏ).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[2, 3, 5]` ra dãy ugly cổ điển; `[2]` ra lũy thừa của 2. Bài 2: mỗi số là tích của số nguyên tố `3` và `5`.
- **Mức 2 (vừa):** Bài 1: `12` và `16`. Bài 2: `[1, 3, 5, 9, 15, 25]`. Bài 3: năm TODO trong comment.
- **Mức 3 (sâu):** Bài 4: heap chứa `(giá_trị, p, chỉ_số)`; sau mỗi lần lấy ra đẩy `(p * ugly[chỉ_số + 1], p, chỉ_số + 1)`. Bài 5: `count_leq` có thể dùng DFS với tập đã thấy; nhị phân tìm `N` nhỏ nhất sao cho `count_leq(N) ≥ n`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert nth_super_ugly_pointers(10, [2, 3, 5]) == 12
assert nth_super_ugly_pointers(5, [2]) == 16

# Bài 2
assert super_ugly_list(6, [3, 5]) == [1, 3, 5, 9, 15, 25]

# Bài 3
def nth_super_ugly_todo_ans(n, primes):
    ugly = [1] * n
    idx = [0] * len(primes)
    for i in range(1, n):
        nxt = min(ugly[idx[j]] * primes[j] for j in range(len(primes)))
        ugly[i] = nxt
        for j in range(len(primes)):
            if ugly[idx[j]] * primes[j] == nxt:
                idx[j] += 1
    return ugly[n - 1]


# Bài 4
def nth_super_ugly_kway2(n, primes):
    ugly = [1]
    heap = [(p, p, 0) for p in primes]
    heapq.heapify(heap)
    while len(ugly) < n:
        val, p, i = heapq.heappop(heap)
        if val != ugly[-1]:
            ugly.append(val)
        heapq.heappush(heap, (p * ugly[i + 1], p, i + 1))
    return ugly[n - 1]


# Bài 5
def count_leq(N, primes):
    return count_super_ugly_leq(N, primes)


def nth_super_ugly_bsearch(n, primes):
    lo, hi = 1, 1
    while count_leq(hi, primes) < n:
        hi *= 2
    while lo < hi:
        mid = (lo + hi) // 2
        if count_leq(mid, primes) >= n:
            hi = mid
        else:
            lo = mid + 1
    return lo


rnd = random.Random(19)
for _ in range(100):
    primes = sorted(rnd.sample(SMALL_PRIMES, rnd.randint(1, 4)))
    n = rnd.randint(1, 40)
    base = nth_super_ugly_pointers(n, primes)
    assert nth_super_ugly_todo_ans(n, primes) == base
    assert nth_super_ugly_kway2(n, primes) == base
    if base <= 3000:
        assert nth_super_ugly_bsearch(n, primes) == base
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (nhiều con trỏ)

```python
class Solution:
    def nthSuperUglyNumber(self, n: int, primes: List[int]) -> int:
        k = len(primes)
        ugly = [1] * n
        idx = [0] * k
        for i in range(1, n):
            nxt = min(ugly[idx[j]] * primes[j] for j in range(k))
            ugly[i] = nxt
            for j in range(k):
                if ugly[idx[j]] * primes[j] == nxt:
                    idx[j] += 1
        return ugly[n - 1]
```

**Checklist**

- [ ] `ugly[0] = 1` (số `1` luôn là số siêu xấu)?
- [ ] Mỗi số nguyên tố có **một con trỏ riêng** trên dãy `ugly`?
- [ ] Tăng **mọi** con trỏ có ứng viên bằng giá trị vừa chọn (khử trùng)?
- [ ] Nếu dùng heap + `break`: `primes` đã sắp tăng dần, và **đẩy `x · p` trước rồi mới `break`**?
- [ ] Trả về `ugly[n - 1]` (chỉ số 0-based)?
- [ ] Độ phức tạp `O(n·k)` (hoặc tốt hơn), không duyệt từng số tự nhiên?
- [ ] Đã thử `n = 1`, một số nguyên tố, `primes = [2,3,5]`, số nguyên tố lớn (`997`)?
- [ ] Đã đối chiếu với oracle duyệt từng số trên input ngẫu nhiên có seed?
- [ ] Skeleton đúng `class Solution` / `nthSuperUglyNumber(self, n: int, primes: List[int]) -> int`?